# DevReady AI — Colab 임시 서버

HuggingFace에서 **EXAONE-Deep-7.8B(4bit) + LoRA 어댑터**를 받아 FastAPI로 띄우고, 공개 URL을 만든다.
AI 관리자 없이 팀원이 **빠르게 테스트**할 때 쓰는 용도. (실제 데모/통합은 RunPod 권장)

> ⚠️ **사전 조건**
> 1. 런타임 → 런타임 유형 변경 → **하드웨어 가속기: GPU(T4)**
> 2. 아래 **설정 셀**에 ngrok authtoken과 어댑터 HF 경로 입력
> 3. 학습한 어댑터(최신본)가 HF에 올라가 있어야 함 — 현재 레포 루트엔 구버전이 있을 수 있으니 `ADAPTER_SUBFOLDER`를 최신 경로로 맞출 것
>
> 추론만 하므로 AI Hub 학습 데이터는 필요 없다.

## 1. 설정

In [ ]:
# ====== 설정 ======
NGROK_TOKEN = ""          # https://dashboard.ngrok.com 에서 발급한 authtoken (필수)
BASE_MODEL  = "LGAI-EXAONE/EXAONE-Deep-7.8B"
BASE_REV    = "e3f42b18f6b1"                       # 베이스 revision 고정 (변경 금지)
ADAPTER_REPO      = "seongchaeae/capstone-interview-ai-lora"
ADAPTER_SUBFOLDER = ""    # 최신 어댑터가 하위폴더면 예: "lora_adapter_v3" / 루트면 "" 

## 2. 패키지 설치
torch는 Colab 기본 설치본을 사용한다(재설치하지 않음 — bitsandbytes 4bit 호환).

In [ ]:
!pip -q install "transformers==4.48.3" "tokenizers==0.21.4" "accelerate==1.2.1" "bitsandbytes==0.45.5" "peft==0.13.2" json-repair fastapi "uvicorn[standard]" pyngrok nest_asyncio

## 3. 모델 로드 (EXAONE 4bit + LoRA 어댑터) — 수 분 소요

In [ ]:
import torch
# 일부 transformers 경로가 참조하는 신규 dtype 호환 shim (구버전 torch 보호)
if not hasattr(torch, "float8_e8m0fnu"):
    torch.float8_e8m0fnu = torch.float32

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

print(">>> 토크나이저 로드...")
tok = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REV, trust_remote_code=True)

print(">>> 베이스 모델 4bit 로드 (수 분)...")
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=BASE_REV,
    quantization_config=bnb, device_map="auto", trust_remote_code=True).eval()

print(">>> LoRA 어댑터 로드...")
kw = {"subfolder": ADAPTER_SUBFOLDER} if ADAPTER_SUBFOLDER else {}
model = PeftModel.from_pretrained(base, ADAPTER_REPO, **kw).eval()
print(">>> 로드 완료. GPU 사용:", round(torch.cuda.memory_allocated()/1024**3, 2), "GB")

## 4. 채점 로직
서버와 동일한 RUBRIC · 한국어 prefill · 강건 파싱. 종합점수(overall)는 모델값을 믿지 않고 **4축 평균으로 재계산**한다.

In [ ]:
import json, re, torch
from json_repair import repair_json

PREFILL = "먼저 지원자의 답변을 평가 항목별로 살펴보겠습니다. "
SCORE_KEYS = ["technical_accuracy", "specificity", "logic", "communication"]

RUBRIC = """당신은 웹개발자 채용 면접관입니다. [면접 질문]에 대한 [지원자 답변]을 평가하세요.

먼저 이 질문이 '기술 질문'인지 '인성·경험·동기 질문'인지 판단하세요.
- 기술 질문(알고리즘, 프로그래밍 언어, 설계, 개발 경험 등): 기술적 정확성을 평가합니다.
- 인성·경험·동기 질문(협업, 갈등 해결, 지원 동기, 부서 배치, 강점/약점 등): 기술 지식을 요구하지 말고, 답변의 태도·경험·사고방식이 타당하고 설득력 있는지를 평가합니다.

각 항목을 0~100점 정수로 채점하세요 (10점 만점이 아닙니다). 모든 항목에 실제 점수를 매기세요.
- technical_accuracy: 내용의 정확성과 타당성
- specificity: 구체성과 깊이
- logic: 논리적 구조
- communication: 전달의 명확성

사고는 한국어로 간결하게 쓰고, 사고 안에는 JSON을 쓰지 마세요.
사고를 마친 뒤 마지막에 아래 JSON 하나만 출력하세요 (JSON 외 텍스트 금지, 모든 내용 한국어):
{"scores":{"technical_accuracy":0,"specificity":0,"logic":0,"communication":0},"strengths":["..."],"improvements":["..."],"feedback":"..."}"""

def _clamp(v):
    try: v = int(round(float(v)))
    except Exception: v = 0
    return max(0, min(100, v))

def parse_eval(gen):
    tail = gen.split("</thought>")[-1] if "</thought>" in gen else gen
    tail = re.sub(r"```(?:json)?", "", tail)
    m = re.search(r"\{.*\}", tail, re.DOTALL)
    if not m: return None
    raw = m.group(0)
    try:
        return json.loads(raw)
    except Exception:
        try:
            o = repair_json(raw, return_objects=True)
            return o if isinstance(o, dict) else None
        except Exception:
            return None

@torch.no_grad()
def evaluate(question, answer):
    user = f"{RUBRIC}\n\n[면접 질문]\n{question}\n\n[지원자 답변]\n{answer}"
    text = tok.apply_chat_template([{"role": "user", "content": user}],
                                   tokenize=False, add_generation_prompt=True) + PREFILL
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    plen = enc["input_ids"].shape[1]
    out = model.generate(**enc, max_new_tokens=2048, do_sample=False)
    gen = tok.decode(out[0][plen:], skip_special_tokens=False)
    obj = parse_eval(gen)
    if not obj or not isinstance(obj.get("scores"), dict):
        return {"ok": False, "error": "채점 JSON 파싱 실패"}
    sc = {k: _clamp(obj["scores"].get(k, 0)) for k in SCORE_KEYS}
    overall = round(sum(sc.values()) / len(sc))   # 모델 종합값 대신 Python 재계산
    return {"ok": True, "scores": sc, "overall": overall,
            "strengths": obj.get("strengths", []),
            "improvements": obj.get("improvements", []),
            "feedback": obj.get("feedback", "")}

# 빠른 자체 점검
print(evaluate("프로세스와 스레드의 차이는?",
               "프로세스는 자원 할당 단위이고, 스레드는 실행 단위로 같은 메모리를 공유합니다."))

## 5. API 서버 + 공개 URL

In [ ]:
import threading, time, uvicorn, nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

nest_asyncio.apply()
app = FastAPI(title="DevReady AI (Colab)")

class EvalReq(BaseModel):
    question: str
    answer: str
    lang: str = "ko"

@app.get("/health")
def health():
    return {"status": "ok", "ready": True, "engine": "EXAONE-Deep-7.8B(4bit)+LoRA", "note": "Colab 임시 서버"}

@app.post("/interview/evaluate")
def do_eval(req: EvalReq):
    return evaluate(req.question, req.answer)

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
public = ngrok.connect(8000)
print(">>> 공개 URL :", public.public_url)
print(">>> 헬스체크 :", public.public_url + "/health")

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True).start()
time.sleep(2)
print(">>> 서버 기동 완료. 위 URL로 호출하세요. (런타임 종료 시 서버도 종료)")

## 6. 호출 예시 / 종료

다른 셀이나 외부에서:
```bash
curl -X POST "<공개URL>/interview/evaluate" \
  -H "Content-Type: application/json" \
  -d '{"question":"REST API란?","answer":"자원을 URI로 표현하고 HTTP 메서드로 다루는 방식입니다."}'
```

종료: 런타임을 끄거나 `ngrok.kill()` 실행. Colab은 일정 시간 후 세션이 끊기니 **임시 테스트용**으로만 쓴다.